In [10]:
# Ô 5.0 — Cosine bằng tay
import numpy as np

v = {"a": np.array([1.0, 2.0, 0.0]), "b": np.array([2.0, 4.0, 0.0]),
     "c": np.array([0.0, 0.0, 3.0]), "d": np.array([-1.0, -2.0, 0.0]),
     "e": np.array([2.0, 1.0, 0.0])}

def cosine(x,y):
  # x @ y là tích vô hướng.
  # np.linalg.norm(x) là độ dài vector.
  return x @ y / (np.linalg.norm(x) * np.linalg.norm(y))

for name in "bcde" :
  print(f"cos(a,{name} = {cosine(v['a'],v[name]):6.3f})")
unit = {k : x / np.linalg.norm(x) for k,x in v.items()} # unit là các vector đã chia cho độ dài của chính nó (đã chuẩn hóa).
# unit = {}
# for k, x in v.items():
#     unit[k] = x / np.linalg.norm(x)
# độ dài ‖a‖ = √(1² + 2² + 0²) = √5 ≈ 2,236

# a / ‖a‖ = [1/2,236,  2/2,236,  0/2,236]
#         = [0,447,    0,894,    0    ]

print("độ dài a sau chuẩn hóa:", round(float(np.linalg.norm(unit["a"])), 3),
      "| tích vô hướng a·e sau chuẩn hóa:", round(float(unit["a"] @ unit["e"]), 3))


cos(a,b =  1.000)
cos(a,c =  0.000)
cos(a,d = -1.000)
cos(a,e =  0.800)
độ dài a sau chuẩn hóa: 1.0 | tích vô hướng a·e sau chuẩn hóa: 0.8


In [16]:
import sys, time
from pathlib import Path
import pandas as pd
sys.path.append(str(Path("..").resolve()))
from src import srs, rag

kb, version = rag.build_kb(set(srs.CODES))
# print(kb)
print("ATT&CK", version, "|", len(kb), "kỹ thuật | thiếu:", sorted(set(srs.CODES) - set(kb["ma"])))
check = kb.merge(srs.WHITELIST, on="ma")
# print(check)
print("tên khác bảng của thầy:", check.loc[check["ten_x"] != check["ten_y"], "ma"].tolist())
print("độ dài mô tả (ký tự):",
      kb["mo_ta"].str.len().describe()[["min", "50%", "max"]].astype(int).to_dict())
print(kb.loc[kb["ma"] == "T1110.001", "mo_ta"].iloc[0][:300])
# kb.loc[ điều_kiện , "mo_ta"] lấy các dòng có True, và chỉ lấy cột mo_ta

ATT&CK 18.1 | 24 kỹ thuật | thiếu: []
tên khác bảng của thầy: []
độ dài mô tả (ký tự): {'min': 535, '50%': 1355, 'max': 3662}
Adversaries with no prior knowledge of legitimate credentials within the system or environment may guess passwords to attempt access to accounts. Without knowledge of the password for an account, an adversary may opt to systematically guess the password using a repetitive or iterative mechanism. An 


In [ ]:
# Ô 5.2 — Tìm bằng TF-IDF (mốc so sánh)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

QUERIES = pd.DataFrame([
    ("many failed SSH logins with different usernames from one IP address", "T1110.001"),
    ("nhiều lần đăng nhập SSH thất bại với các tên người dùng khác nhau từ một địa chỉ IP",
     "T1110.001"),
    ("xóa nhật ký sự kiện bảo mật của Windows để che dấu vết", "T1070.001"),
    ("attacker encrypted files on the file server and left a ransom note", "T1486"),
    ("tải một công cụ từ máy chủ bên ngoài về thư mục /tmp bằng wget", "T1105"),
    ("tắt tính năng bảo vệ thời gian thực của phần mềm diệt virus", "T1562.001"),
], columns=["cau_hoi", "ky_vong"])

docs = (kb["ma"] + " " + kb["ten"] + ". " + kb["mo_ta"]).tolist()
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit(docs)
S_tfidf = cosine_similarity(tfidf.transform(QUERIES["cau_hoi"]), tfidf.transform(docs))
# Dòng này tính bảng độ giống cosine giữa mọi câu hỏi và mọi tài liệu kỹ thuật,

def top_k(S, k=3):
    return [[kb["ma"][j] for j in row.argsort()[::-1][:k]] for row in S]

QUERIES["tfidf_top3"] = top_k(S_tfidf)
QUERIES["tfidf_max"] = S_tfidf.max(axis=1).round(3) # Độ giống cao nhất của câu hỏi đó
QUERIES["tfidf_hit3"] = [e in t for e, t in zip(QUERIES["ky_vong"], QUERIES["tfidf_top3"])] # True nếu mã kỳ vọng nằm trong top 3
print(QUERIES[["ky_vong", "tfidf_top3", "tfidf_max", "tfidf_hit3"]].to_string())

["T1003.001 LSASS Memory. Adversaries may attempt to access credential material stored in the process memory of the Local Security Authority Subsystem Service (LSASS). After a user logs on, the system generates and stores a variety of credential materials in LSASS process memory. These credential materials can be harvested by an administrative user or SYSTEM and used to conduct Lateral Movement using Use Alternate Authentication Material. As well as in-memory techniques, the LSASS process memory can be dumped from the target host and analyzed on a local system. For example, on the target host use procdump: * procdump -ma lsass.exe lsass_dump Locally, mimikatz can be run using: * sekurlsa::Minidump lsassdump.dmp * sekurlsa::logonPasswords Built-in Windows tools such as `comsvcs.dll` can also be used: * rundll32.exe C:\\Windows\\System32\\comsvcs.dll MiniDump PID lsass.dmp full Similar to Image File Execution Options Injection, the silent process exit mechanism can be abused to create a 

In [ ]:
# Ô 5.3 — Embedding: kiểm độ dài và nghĩa Anh–Việt
sentences = ["SSH brute force from one IP address",
             "dò mật khẩu SSH liên tục từ một địa chỉ IP",
             "xóa nhật ký sự kiện Windows",
             "the attacker cleared the Windows event log"]
t0 = time.time()
E = rag.embed([rag.as_query(s) for s in sentences])
print(E)
# A = np.array([[1, 2, 3],
#               [4, 5, 6]])

# A.sum()           # 21: cộng TẤT CẢ
# A.sum(axis=0)     # [5 7 9]: mỗi CỘT một tổng (1+4, 2+5, 3+6)
# A.sum(axis=1)     # [ 6 15]: mỗi HÀNG một tổng (1+2+3, 4+5+6)
print(E.shape, "| độ dài từng vector:", np.linalg.norm(E, axis=1).round(3),
      "| giây:", round(time.time() - t0, 2))
print(pd.DataFrame(E @ E.T, index=range(4), columns=range(4)).round(3).to_string()) # E @ E.T là ma trận 4×4; mỗi ô là độ giống giữa hai câu.

[[-0.13816921 -0.04241702 -0.02150596 ... -0.07312128  0.00167438
  -0.03622845]
 [-0.11730626 -0.07245269  0.02447435 ... -0.02612847 -0.01429921
  -0.03073552]
 [-0.19262089  0.02269783  0.03562111 ...  0.02491019  0.01321885
  -0.04583387]
 [-0.16454296  0.01297735  0.03394034 ...  0.0398482  -0.04929933
  -0.03218696]]
(4, 768) | độ dài từng vector: [1. 1. 1. 1.] | giây: 2.37
       0      1      2      3
0  1.000  0.700  0.131  0.303
1  0.700  1.000  0.125  0.200
2  0.131  0.125  1.000  0.673
3  0.303  0.200  0.673  1.000


In [23]:
# Ô 5.4 — Embed kho và lưu vào pgvector
t0 = time.time()
doc_vecs = rag.embed([rag.as_doc(f"{r.ma} {r.ten}", r.mo_ta[:2000]) for r in kb.itertuples()])
print(doc_vecs)
print(doc_vecs.shape, "|", round(time.time() - t0, 1), "giây")
conn = rag.connect()
rag.create_table(conn)
rag.upsert(conn, kb, doc_vecs, version)
print(conn.execute("SELECT count(*), min(attack_ver), min(embed_model) FROM attack_kb").fetchone())
print(conn.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'").fetchone())

[[-0.07441168 -0.01573983 -0.0502682  ...  0.03904613 -0.0175935
  -0.0456403 ]
 [-0.05399181 -0.01552757 -0.01921274 ... -0.05278795  0.01140323
  -0.03140584]
 [-0.09648018 -0.05671278 -0.02012884 ... -0.04682252  0.00426067
   0.01509954]
 ...
 [-0.04326739 -0.00796676 -0.00177234 ...  0.01427492 -0.04634586
  -0.01384909]
 [-0.05901243 -0.05011791  0.01150538 ... -0.00529019  0.0275464
  -0.06297782]
 [-0.04991068 -0.03684785 -0.03182902 ...  0.00514492  0.00847537
  -0.04430626]]
(24, 768) | 22.4 giây
(24, '18.1', 'embeddinggemma')
('0.8.7',)


In [24]:
# Ô 5.5 — Tìm bằng pgvector, so với TF-IDF
q_vecs = rag.embed([rag.as_query(q) for q in QUERIES["cau_hoi"]])
found = [rag.search(conn, vec, k=3) for vec in q_vecs]
QUERIES["emb_top3"] = [[ma for ma, ten, giong in rows] for rows in found]
QUERIES["emb_max"] = [round(float(rows[0][2]), 3) for rows in found]
QUERIES["emb_hit3"] = [e in t for e, t in zip(QUERIES["ky_vong"], QUERIES["emb_top3"])]
print(QUERIES[["ky_vong", "tfidf_top3", "emb_top3", "emb_max"]].to_string())
print("đúng trong top 3 | TF-IDF:", int(QUERIES["tfidf_hit3"].sum()), "/", len(QUERIES),
      "| embedding:", int(QUERIES["emb_hit3"].sum()), "/", len(QUERIES))

     ky_vong                         tfidf_top3                           emb_top3  emb_max
0  T1110.001  [T1021.004, T1566.001, T1110.001]  [T1021.004, T1110.001, T1136.001]    0.401
1  T1110.001      [T1021.004, T1046, T1059.004]  [T1021.004, T1110.001, T1136.001]    0.402
2  T1070.001  [T1053.005, T1070.001, T1547.001]  [T1070.001, T1562.001, T1547.001]    0.620
3      T1486          [T1486, T1027, T1505.003]          [T1486, T1027, T1566.001]    0.547
4      T1105      [T1105, T1566.001, T1548.003]      [T1105, T1505.003, T1059.001]    0.441
5  T1562.001  [T1562.001, T1566.001, T1548.003]  [T1562.001, T1070.001, T1547.001]    0.388
đúng trong top 3 | TF-IDF: 5 / 6 | embedding: 6 / 6


In [25]:
# Ô 5.6 — Kiểm lại bằng numpy và xem kế hoạch truy vấn
same = top_k(q_vecs @ doc_vecs.T) == QUERIES["emb_top3"].tolist()
print("pgvector và numpy cho cùng thứ hạng:", same)
plan = conn.execute("""
    EXPLAIN SELECT ma FROM attack_kb
    ORDER BY embedding <=> (SELECT embedding FROM attack_kb WHERE ma = 'T1486')
    LIMIT 3""").fetchall()
print("\n".join(row[0] for row in plan))
QUERIES.to_csv("../results/c3_b05_retrieval.csv", index=False)

pgvector và numpy cho cùng thứ hạng: True
Limit  (cost=26.30..26.31 rows=3 width=40)
  InitPlan 1
    ->  Index Scan using attack_kb_pkey on attack_kb attack_kb_1  (cost=0.15..8.17 rows=1 width=32)
          Index Cond: (ma = 'T1486'::text)
  ->  Sort  (cost=18.14..18.94 rows=320 width=40)
        Sort Key: ((attack_kb.embedding <=> (InitPlan 1).col1))
        ->  Seq Scan on attack_kb  (cost=0.00..14.00 rows=320 width=40)
